# KernelFuse Phase 4 — Nsight on Tier B

**Goal:** DRAM throughput + warp stall reasons for fused / smem / vec4.

- **T4 (sm_75):** same 64 KiB/SM cliff as the GTX 1650 → occupancy probe should show **~3 vs 1** blocks at 4096/8192 (matches Phase 3 regression story).
- **A100 (sm_80):** great for Nsight counters and the memory-bound / vec4 story, but ~164–192 KiB smem/SM means **both widths stay high-occupancy** — do not expect the 2.7× smem BW drop here.

**Setup:** Runtime → GPU. Upload `kernelfuse_upload.zip` to `/content`, then run all cells. Fallback: clone `KashMaj1708/KernelFuse`.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total,clocks.sm,clocks.mem,temperature.gpu --format=csv
!nvcc --version | tail -n 1
!which ncu || ls /usr/local/cuda*/bin/ncu 2>/dev/null; ls /opt/nvidia/nsight-compute/*/ncu 2>/dev/null | head

In [ ]:
# Prefer kernelfuse_upload.zip in /content; else clone from GitHub.
import os
import shutil
import zipfile
from pathlib import Path

ROOT = Path("/content/KernelFuse")
ZIP_CANDIDATES = [
    Path("/content/kernelfuse_upload.zip"),
    Path("/content/KernelFuse/kernelfuse_upload.zip"),
]
REPO = "https://github.com/KashMaj1708/KernelFuse.git"

def has_sources(root: Path) -> bool:
    return (root / "kernels/rmsnorm/rmsnorm_fused_smem.cu").is_file() and (
        root / "scripts/run_phase4_profile.sh"
    ).is_file()

if has_sources(ROOT):
    print("using existing", ROOT)
else:
    zpath = next((z for z in ZIP_CANDIDATES if z.is_file()), None)
    if zpath is not None:
        print("extracting", zpath)
        if ROOT.exists():
            shutil.rmtree(ROOT)
        with zipfile.ZipFile(zpath, "r") as zf:
            zf.extractall("/content")
        # Zip root is KernelFuse/...
        if not has_sources(ROOT):
            raise SystemExit(f"zip extracted but sources missing under {ROOT}")
    else:
        print("no zip found; cloning", REPO)
        if ROOT.exists():
            shutil.rmtree(ROOT)
        !git clone --depth 1 {REPO} {ROOT}

os.chdir(ROOT)
print("cwd", Path.cwd())
print("ok:", has_sources(ROOT))

In [ ]:
import os
import subprocess
from pathlib import Path

cap = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
    text=True,
).strip().splitlines()[0].strip()
major, minor = cap.split(".")
arch = f"sm_{major}{minor}"
print(f"compute_cap={cap} -> CUDA_ARCH={arch}")
if major in ("8", "9"):
    print(
        "NOTE: high smem/SM — cols=8192 will NOT reproduce the 1650/T4 occupancy cliff. "
        "Still collect DRAM + stall metrics; treat occupancy vs Phase 3 as qualitative only."
    )

env = os.environ.copy()
env["CUDA_ARCH"] = arch
script = Path("scripts/run_phase4_profile.sh")
assert script.is_file(), script
subprocess.run(["chmod", "+x", str(script)], check=True)
raise SystemExit(subprocess.run(["bash", str(script)], env=env).returncode)

In [ ]:
from pathlib import Path
import pandas as pd

out = Path("reports/phase4")
print((out / "occupancy_probe.txt").read_text() if (out / "occupancy_probe.txt").exists() else "no occupancy yet")
for p in sorted(out.glob("*.csv")):
    print("\n===", p.name, "===")
    try:
        df = pd.read_csv(p, skiprows=0)
        # ncu CSV layout varies; show metric-ish columns if present
        cols = [c for c in df.columns if any(k in c.lower() for k in ("metric", "value", "kernel", "dram", "warp", "occup"))]
        display(df[cols].head(40) if cols else df.head(20))
    except Exception as e:
        print(p, e)
        print(p.read_text()[:2000])

## What to paste into `docs/phase_4_report.md`

1. Device name + compute capability + clocks before/after.
2. Occupancy probe blocks/SM at 4096 vs 8192:
   - T4/sm_75: expect **~3 vs 1** (matches 1650).
   - A100: expect **both high** — say so explicitly; cliff is Tier-A/Turing-only.
3. Table: achieved occupancy %, `dram__bytes.sum.per_second`, barrier stall %, long_scoreboard stall % for `smem_4096`, `smem_8192`, `vec4_8192`, `fused_8192`.
4. On A100 focus: memory-bound confirmation + why vec4 beats scalar smem at equal occupancy; do not claim the 2.7× regression reproduced.
5. Download `reports/phase4/` back to the laptop repo.